# Population loader

Loads World Bank total population, indicator `SP.POP.TOTL`, into the MySQL
`population` table.

**Period:** 2020 through the current year. As of now this means 2020-2026.

Same shape as the GDP loader, and deliberately so - both read one World Bank
indicator into one table, and the shared work lives in `etl.py`. The only real
differences are the indicator code and the cast: population is a count of
people, so `int`, while GDP is money, so `Decimal`.

The notebook:
- requests the indicator for all countries/economies
- keeps only economies that exist in `dim_country`
- stores `country_iso3` alongside the country label
- skips unpublished `NULL` observations and counts them
- uses an upsert so the notebook can be rerun safely
- prints a year-level validation summary after loading

Run `project_1.sql` and `dimensions.ipynb` first: the foreign key
`fk_population_country` needs `dim_country` populated.

MySQL credentials are read from the existing `.env` file.

In [1]:
%pip install -q requests python-dotenv mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datetime import date

import etl


# ============================================================
# CONFIGURATION
# ============================================================

START_YEAR = 2020
END_YEAR = date.today().year   # 2026 now; updates automatically in future years

INDICATOR = "SP.POP.TOTL"      # Population, total


def to_people(value):
    """
    Population is a headcount, so BIGINT UNSIGNED and a plain int.

    The World Bank returns these as floats; int() drops the always-zero
    fractional part rather than storing a decimal that cannot exist.
    """
    return int(value)


# ============================================================
# UPSERT
#
# The unique key is (year, country_iso3), so a rerun refreshes the value and
# the country label instead of inserting a duplicate.
#
# The aliased-row form (AS new / new.col) rather than VALUES(): MySQL
# deprecated VALUES() inside ON DUPLICATE KEY UPDATE in 8.0.20. Both forms are
# collapsed into a single multi-row statement by mysql-connector's
# executemany - measured on a 5,000-row load, not assumed.
# ============================================================

INSERT_QUERY = """
INSERT INTO population
    (`year`, country, country_iso3, population)
VALUES
    (%s, %s, %s, %s) AS new
ON DUPLICATE KEY UPDATE
    country    = new.country,
    population = new.population
"""

In [3]:
# ============================================================
# EXTRACT -> TRANSFORM -> LOAD
# ============================================================

session = etl.make_session()
connection = etl.connect_mysql()

try:
    valid_iso3 = etl.load_country_keys(connection)
    print("Countries available in dim_country:", len(valid_iso3))

    if not valid_iso3:
        raise RuntimeError(
            "dim_country is empty - run project_1.sql and dimensions.ipynb first."
        )

    # ----------------------------
    # EXTRACT
    # ----------------------------
    api_rows = etl.fetch_worldbank_indicator(
        session, INDICATOR, START_YEAR, END_YEAR
    )
    print("Population observations returned by API:", len(api_rows))

    # ----------------------------
    # TRANSFORM
    # ----------------------------
    population_rows, skipped_null, skipped_unknown = etl.transform_indicator(
        api_rows, valid_iso3, to_people
    )

    print("Population rows ready for MySQL:", len(population_rows))
    print("Skipped - value not published yet (NULL):", skipped_null)
    print("Skipped - aggregate or unknown economy:", skipped_unknown)

    # ----------------------------
    # LOAD
    # ----------------------------
    loaded = etl.upsert(connection, INSERT_QUERY, population_rows)
    print("\nPopulation load completed successfully. Rows processed:", loaded)

except Exception:
    print("\nPopulation load failed.")
    raise

finally:
    connection.close()
    print("MySQL connection closed.")

Countries available in dim_country: 217
Population observations returned by API: 1590
Population rows ready for MySQL: 1302
Skipped - value not published yet (NULL): 0
Skipped - aggregate or unknown economy: 288

Population load completed successfully. Rows processed: 1302
MySQL connection closed.


In [4]:
# ============================================================
# VALIDATION
#
# Population is expected to be a complete grid: every country, every year.
# If countries_per_year ever stops being constant, the source changed.
# ============================================================

connection = etl.connect_mysql()

try:
    print("Population rows currently stored by year:")
    etl.print_rows(connection, """
        SELECT
            `year`,
            COUNT(*)         AS countries,
            MIN(population)  AS min_population,
            MAX(population)  AS max_population
        FROM population
        GROUP BY `year`
        ORDER BY `year`
    """)

    print("\nGrid completeness (distinct countries x distinct years vs rows):")
    etl.print_rows(connection, """
        SELECT
            COUNT(DISTINCT country_iso3)                     AS countries,
            COUNT(DISTINCT `year`)                           AS years,
            COUNT(DISTINCT country_iso3) * COUNT(DISTINCT `year`) AS expected_rows,
            COUNT(*)                                         AS actual_rows
        FROM population
    """)

finally:
    connection.close()

Population rows currently stored by year:
   (2020, 217, 10399, 1411100000)
   (2021, 217, 10194, 1414203896)
   (2022, 217, 9992, 1425423212)
   (2023, 217, 9816, 1438069596)
   (2024, 217, 9646, 1450935791)
   (2025, 217, 9492, 1463865525)

Grid completeness (distinct countries x distinct years vs rows):
   (217, 6, 1302, 1302)
